# 🧪 Linting System Tests

### 🔁 Step 1: Set up Environment

In [1]:
from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")

Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt 'generate' with ID: 1 and GUID: 709dd9fe-bbdc-4e7d-95e8-311a15c70e16
Seeded AgentPrompt 'linting_generator_agent' with ID: 2 and GUID: 2d530a1e-083b-443c-bbc3-d29bf7fe88c4
Seeded SystemPrompt 'format' with ID: 1 and GUID: fe7ad2a6-0f44-4969-9213-8d688944f4f4
Seeded SystemPrompt 'linting_system' with ID: 2 and GUID: 901e9f2a-d800-4a86-bf89-b44429fbf4ef
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
✅ Seeded state provider configurations successfully.
Seeded system provider configurations successfully.
✅ Seeded controller providers
✅ Seeded program providers
✅ Seeded session configs
✅ Loaded environment variables from env/.env


### 🔍 Step 2: Create Basic Prompt

In [2]:
from sqlalchemy.orm import Session
from app.db.models import PromptProviderConfig, AgentPrompt, SystemPrompt

# Validate Prompt Providers and Prompts
with Session(bind=engine) as session:
    print("\n🔍 PromptProviderConfig records:")
    providers = session.query(PromptProviderConfig).all()
    for provider in providers:
        print(f"ID: {provider.id}, Name: {provider.name}, Artifact Path: {provider.artifact_path}, Config: {provider.config}")

    print("\n🔍 AgentPrompt records:")
    agent_prompts = session.query(AgentPrompt).all()
    for prompt in agent_prompts:
        print(f"ID: {prompt.id}, Name: {prompt.name}, Artifact Path: {prompt.artifact_path}")

    print("\n🔍 SystemPrompt records:")
    system_prompts = session.query(SystemPrompt).all()
    for prompt in system_prompts:
        print(f"ID: {prompt.id}, Name: {prompt.name}, Artifact Path: {prompt.artifact_path}")


🔍 PromptProviderConfig records:
ID: 1, Name: basic_prompt_provider, Artifact Path: 75f652ad-03c6-4b5f-8b0f-001d151a7c4e.py, Config: {}
ID: 2, Name: linting_prompt_provider, Artifact Path: 20542dd5-6cff-4572-b5e1-818374b5f090.py, Config: {'agent_prompt_id': 2, 'system_prompt_id': 2, 'context_provider_id': 2}

🔍 AgentPrompt records:
ID: 1, Name: generate, Artifact Path: 709dd9fe-bbdc-4e7d-95e8-311a15c70e16.txt
ID: 2, Name: linting_generator_agent, Artifact Path: 2d530a1e-083b-443c-bbc3-d29bf7fe88c4.txt

🔍 SystemPrompt records:
ID: 1, Name: format, Artifact Path: fe7ad2a6-0f44-4969-9213-8d688944f4f4.txt
ID: 2, Name: linting_system, Artifact Path: 901e9f2a-d800-4a86-bf89-b44429fbf4ef.txt


### 🔍 Step 3: Test ToolProvider

In [3]:
from app.factories.tool_provider_factory import ToolProviderFactory
from app.db.models import ProviderLog
from sqlalchemy.orm import Session
from pathlib import Path

# 🧪 Step 1: Create small Python test file
test_file = Path("tests/example_black_target.py")
test_file.parent.mkdir(parents=True, exist_ok=True)
test_file.write_text("def add( a, b ):\n return a + b\n", encoding="utf-8")

# 🔌 Step 2: Instantiate the black tool provider
black_tool = ToolProviderFactory.create(id=1)

# 🚀 Step 3: Run it
result = black_tool.run(input={"target": str(test_file)}, session_id="black-test-session")
print("🧪 Result from BlackToolProvider:")
print(result)

# 📜 Step 4: Query provider logs
with Session(bind=engine) as session:
    logs = session.query(ProviderLog).order_by(ProviderLog.timestamp.desc()).all()
    print("\n🗂️ Provider Logs:")
    for log in logs:
        print(f"[{log.timestamp}] {log.provider_type} (ID {log.provider_id})")
        print(f"Input: {log.input}")
        print(f"Output: {log.output}")
        print("-" * 80)


🧪 Result from BlackToolProvider:
return_code=0 stdout='' stderr='' violations=None metrics=None summary='Black formatting applied'

🗂️ Provider Logs:
[2025-06-02 17:22:31.315167+00:00] PROVIDER_TYPE.TOOL (ID 1)
Input: {"target": "tests\\example_black_target.py"}
Output: {"return_code": 0, "stdout": "", "stderr": "", "violations": null, "metrics": null, "summary": "Black formatting applied"}
--------------------------------------------------------------------------------


### 🔍 Step 4: Test Score Provider

In [4]:
from app.factories.score_provider_factory import ScoreProviderFactory
from app.db.models import ProviderLog
from sqlalchemy.orm import Session
from pathlib import Path

# 🔧 Create test file
test_file = Path("tests/example_lint_target.py")
test_file.parent.mkdir(parents=True, exist_ok=True)
test_file.write_text("def lint_test( a,b ):\n return a+b\n", encoding="utf-8")

# 🔌 Instantiate the score provider (linting)
score_provider = ScoreProviderFactory.create(id=1)

# 🚀 Run it
score_output = score_provider.run(input={"file_path": str(test_file)}, session_id="score-test-session")

print("🧠 Linting Score Output:")
print(score_output)

# 📜 Query provider logs
with Session(bind=engine) as session:
    logs = session.query(ProviderLog).order_by(ProviderLog.timestamp.desc()).all()
    print("\n🗂️ Provider Logs:")
    for log in logs:
        print(f"Session id: {log.session_id}")
        print(f"[{log.timestamp}] {log.provider_type} (ID {log.provider_id})")
        print(f"Input: {log.input}")
        print(f"Output: {log.output}")
        print("-" * 80)


🧠 Linting Score Output:
name=<SCORING_METRIC_TYPE.LINTING_SCORE: 'linting_score'> value=0.0 components={'ruff': 0.0, 'black': 0.0, 'mypy': 0.0} summary='No violations found.'

🗂️ Provider Logs:
Session id: score-test-session
[2025-06-02 17:22:31.951698+00:00] PROVIDER_TYPE.SCORE (ID 1)
Input: {"file_path": "tests\\example_lint_target.py"}
Output: {"name": "linting_score", "value": 0.0, "components": {"ruff": 0.0, "black": 0.0, "mypy": 0.0}, "summary": "No violations found."}
--------------------------------------------------------------------------------
Session id: score-test-session
[2025-06-02 17:22:31.932526+00:00] PROVIDER_TYPE.TOOL (ID 5)
Input: {"target": "tests\\example_lint_target.py"}
Output: {"return_code": 0, "stdout": "\u001b\u001bSuccess: no issues found in 1 source file\u001b\n", "stderr": "", "violations": null, "metrics": null, "summary": "Mypy check passed"}
--------------------------------------------------------------------------------
Session id: score-test-session

### 🔍 Step 5: Test Context Provider

In [5]:
from dataclasses import asdict
from app.factories.context_provider_factory import ContextProviderFactory
from app.db.models import ProviderLog
from sqlalchemy.orm import Session
from pathlib import Path
import json

# 🔧 Create a test Python file
test_file = Path("tests/context_example.py")
test_file.parent.mkdir(parents=True, exist_ok=True)
test_file.write_text("def example( a, b ):\n return a + b\n", encoding="utf-8")

# 🧠 Instantiate the context provider (linting)
context_provider = ContextProviderFactory.create(id=2)

# 🚀 Run it
context = context_provider.run(
    input={
        "file_path": str(test_file),
        "session_id": "context-test-session",
        "system": "linting"
    }
)

print("🧠 Context Output:")
print(json.dumps(context.model_dump(), indent=2))

# 📜 Query and print provider logs
with Session(bind=engine) as session:
    logs = session.query(ProviderLog).order_by(ProviderLog.timestamp.desc()).all()
    print("\n📚 Provider Logs:")
    for log in logs:
        print(f"[{log.timestamp}] {log.provider_type} (ID {log.provider_id})")
        print(f"Input: {log.input}")
        print(f"Output: {log.output}")
        print("-" * 80)


🧠 Context Output:
{
  "context": {
    "file_path": "C:\\Repos\\codecritic\\tests\\context_example.py",
    "source_code": "def example( a, b ):\n return a + b\n",
    "score": {
      "name": "linting_score",
      "value": 0.0,
      "components": {
        "ruff": 0.0,
        "black": 0.0,
        "mypy": 0.0
      },
      "summary": "No violations found."
    },
    "conversation_log": []
  },
  "summary": "Context for context_example.py, 0.0 score, 0 log entries"
}

📚 Provider Logs:
[2025-06-02 17:22:32.687022+00:00] PROVIDER_TYPE.CONTEXT (ID 2)
Input: {"file_path": "tests\\context_example.py", "session_id": "context-test-session", "system": "linting"}
Output: {"context": {"file_path": "C:\\Repos\\codecritic\\tests\\context_example.py", "source_code": "def example( a, b ):\n return a + b\n", "score": {"name": "linting_score", "value": 0.0, "components": {"ruff": 0.0, "black": 0.0, "mypy": 0.0}, "summary": "No violations found."}, "conversation_log": []}, "summary": "Context for 

### 🔍 Step 6: Test Prompt Provider

In [6]:
from app.factories.prompt_provider_factory import PromptProviderFactory
from app.db.models import ProviderLog
from sqlalchemy.orm import Session
from pathlib import Path
import json

# 🔧 Create a small test file to trigger prompt generation
test_file = Path("tests/prompt_context_example.py")
test_file.parent.mkdir(parents=True, exist_ok=True)
test_file.write_text("def greet( name ):\n return f\"Hello, {name}!\"\n", encoding="utf-8")

# 🧠 Instantiate the prompt provider (ID = 2, uses context provider = 2)
prompt_provider = PromptProviderFactory.create(id=2)

# 🚀 Run it
final_prompt = prompt_provider.run(
    input={
        "file_path": str(test_file),
        "session_id": "prompt-test-session",
        "system": "linting"
    }
)

print("🧠 Final Prompt Output:")
print(final_prompt)

# 📜 Query provider logs
with Session(bind=engine) as session:
    logs = session.query(ProviderLog).order_by(ProviderLog.timestamp.desc()).all()
    print("\n📚 Provider Logs:")
    for log in logs:
        print(f"[{log.timestamp}] {log.provider_type} (ID {log.provider_id})")
        print(f"Input: {log.input}")
        print(f"Output: {log.output}")
        print("-" * 80)


🧠 Final Prompt Output:
prompt='\n            You are the Generator Agent in the Linting System.\n\nYou specialize in incrementally improving Python code quality by addressing lint violations, enhancing readability, and promoting best practices.\n\nYour strategy is:\n\nFocus on minimal, targeted improvements that preserve original structure\n\nApply type hints and clarify variable names where appropriate\n\nUse reasoning to resolve tool-reported issues when they conflict\n\nYou will be evaluated by a Discriminator Agent and may be prompted to revise your output. If you are in a retry round, incorporate prior feedback before retrying.\n\nMaintain a helpful, confident tone in your comments. Your improvements should be easy for a human developer to understand and accept.\n\n🚫 Your response must strictly follow this format:\n\n[CODE]\n# your revised Python code here\n[/CODE]\n\n[CONVERSATION_LOG_ENTRY]\n# bullet list of what changed, why, and tradeoffs if any\n[/CONVERSATION_LOG_ENTRY]\n\n⚠

### 🔍 Step 7: Test Generator Provider

In [7]:
from app.factories.agent_provider_factory import AgentProviderFactory
from app.db.models import ProviderLog
from sqlalchemy.orm import Session
from pathlib import Path

# 🔧 Create a test file
test_file = Path("tests/agent_generator_example.py")
test_file.parent.mkdir(parents=True, exist_ok=True)
test_file.write_text("def greet( name ):\n return f\"Hello, {name}!\"\n", encoding="utf-8")

# 🧠 Run generator agent (ID = 2)
generator = AgentProviderFactory.create(id=2)

generator_output = generator.run(
    input={
        "file_path": str(test_file),
        "session_id": "agent-test-session",
        "system": "linting"
    }
)

print("🧠 Generator Output:")
print(generator_output)

# 📜 Print provider logs
with Session(bind=engine) as session:
    logs = session.query(ProviderLog).order_by(ProviderLog.timestamp.desc()).all()
    print("\n📚 Provider Logs:")
    for log in logs:
        print(f"[{log.timestamp}] {log.provider_type} (ID {log.provider_id})")
        print(f"Input: {log.input}")
        print(f"Output: {log.output}")
        print("-" * 80)


🧠 Generator Output:
response='[CODE]\ndef greet(name: str) -> str:\n    return f"Hello, {name}!"\n[/CODE]\n\n[CONVERSATION_LOG_ENTRY]\n- Removed unnecessary spaces around the parameter in the function definition to conform to PEP8 guidelines.\n- Added type hints for the function parameter and return type to improve code clarity and maintainability.\n- Ensured the function remains simple and readable, preserving its original intent and structure.\n[/CONVERSATION_LOG_ENTRY]' log='- Removed unnecessary spaces around the parameter in the function definition to conform to PEP8 guidelines.\n- Added type hints for the function parameter and return type to improve code clarity and maintainability.\n- Ensured the function remains simple and readable, preserving its original intent and structure.' decision='unknown' snapshot_id='3dc68552-769f-4baf-9aaf-a0e338811fa1'

📚 Provider Logs:
[2025-06-02 17:22:38.481744+00:00] PROVIDER_TYPE.AGENT (ID 2)
Input: {"file_path": "tests\\agent_generator_exampl

### 🔍 Step 8: Test Discriminator Provider

In [8]:
from app.factories.agent_provider_factory import AgentProviderFactory
from app.db.models import ProviderLog, AgentConversationLog
from sqlalchemy.orm import Session

# 🧪 Run discriminator agent (ID = 3) using same session
discriminator = AgentProviderFactory.create(id=3)

discriminator_output = discriminator.run(
    input={
        "system": "linting",
        "session_id": "agent-test-session"
    }
)

print("🧠 Discriminator Output:")
print(discriminator_output)

# 📜 Print provider logs
with Session(bind=engine) as session:
    logs = session.query(ProviderLog).order_by(ProviderLog.timestamp.desc()).all()
    print("\n📚 Provider Logs:")
    for log in logs:
        print(f"[{log.timestamp}] {log.provider_type} (ID {log.provider_id})")
        print(f"Input: {log.input}")
        print(f"Output: {log.output}")
        print("-" * 80)

    # 💬 Print agent conversation log
    convos = session.query(AgentConversationLog).order_by(AgentConversationLog.timestamp.desc()).all()
    print("\n🗣️ Agent Conversation Log:")
    for convo in convos:
        print(f"[{convo.timestamp}] {convo.agent_type} said:\n{convo.content}\n" + "-" * 80)


🧠 Discriminator Output:
response='[AGENT_DECISION]accept[/AGENT_DECISION]' log=None decision='unknown' snapshot_id=None

📚 Provider Logs:
[2025-06-02 17:22:38.517763+00:00] PROVIDER_TYPE.AGENT (ID 3)
Input: {"system": "linting", "session_id": "agent-test-session"}
Output: {"response": "[AGENT_DECISION]accept[/AGENT_DECISION]", "log": null, "decision": "unknown", "snapshot_id": null}
--------------------------------------------------------------------------------
[2025-06-02 17:22:38.481744+00:00] PROVIDER_TYPE.AGENT (ID 2)
Input: {"file_path": "tests\\agent_generator_example.py", "session_id": "agent-test-session", "system": "linting"}
Output: {"response": "[CODE]\ndef greet(name: str) -> str:\n    return f\"Hello, {name}!\"\n[/CODE]\n\n[CONVERSATION_LOG_ENTRY]\n- Removed unnecessary spaces around the parameter in the function definition to conform to PEP8 guidelines.\n- Added type hints for the function parameter and return type to improve code clarity and maintainability.\n- Ensured 

### 🔍 Step 9: Test State Provider

In [9]:
# 🔁 State Provider Test Cell for Generator and Discriminator

from app.factories.state_provider_factory import StateProviderFactory
from app.db.models import StateTransitionLog
from sqlalchemy.orm import Session
from pathlib import Path
import json

# Test file for generator agent
test_file = Path("tests/state_generator_example.py")
test_file.parent.mkdir(parents=True, exist_ok=True)
test_file.write_text("def greet( name ):\n return f\"Hello, {name}!\"\n", encoding="utf-8")

# Shared state
initial_state = {
    "file_path": str(test_file),
    "session_id": "state-test-session",
    "system": "linting",
    "state": "start"
}

# 🧠 Run Generator State Provider (ID = 1)
generator_state = StateProviderFactory.create(id=1)
after_generate = generator_state.run(initial_state)

print("🌀 Generator State Output:")
print(json.dumps(after_generate.model_dump(), indent=2))

# 🧠 Reset for Discriminator State Provider
discriminator_input = {
    **after_generate.model_dump(),
    "state": "start",
    "session_id": "agent-test-session"  # ✅ Required for logging
}
discriminator_input.pop("reason", None)
discriminator_input.pop("_last_state", None)
discriminator_input.pop("output", None)

# 🧠 Run Discriminator State Provider (ID = 2)
discriminator_state = StateProviderFactory.create(id=2)
after_discriminate = discriminator_state.run(discriminator_input)

print("\n🌀 Discriminator State Output:")
print(json.dumps(after_discriminate.model_dump(), indent=2))

# 🧾 Print state transition logs
print("\n📚 State Transitions:")
with Session(bind=engine) as session:
    logs = session.query(StateTransitionLog).order_by(StateTransitionLog.timestamp.desc()).all()
    for log in logs[:5]:  # Just show last 5 for brevity
        print(f"[{log.timestamp}] {log.from_state} → {log.to_state} | Reason: {log.reason}")


🌀 Generator State Output:
{
  "state": "end",
  "previous_state": "generate",
  "state_type": "end",
  "reason": "success",
  "decision": "final",
  "steps": 2,
  "max_steps": 20,
  "summary": "generation complete",
  "output": {
    "state": "end",
    "file_path": "tests\\state_generator_example.py",
    "session_id": "state-test-session",
    "system": "linting",
    "steps": 2,
    "reason": "generation complete",
    "_last_state": "generate",
    "agent_output": {
      "response": "[CODE]\ndef greet(name: str) -> str:\n    return f\"Hello, {name}!\"\n[/CODE]\n\n[CONVERSATION_LOG_ENTRY]\n- Removed unnecessary spaces around the parameter in the function definition to comply with PEP8 guidelines.\n- Added type hints to the function parameters and return type to improve code clarity and maintainability.\n- Ensured the function remains simple and readable, preserving its original intent.\n[/CONVERSATION_LOG_ENTRY]",
      "log": "- Removed unnecessary spaces around the parameter in t

### 🔍 Step 10: Test System Provider

In [10]:
# 🧪 Linting SystemProvider Test

from app.factories.system_provider_factory import SystemProviderFactory
from app.db.models import StateTransitionLog
from sqlalchemy.orm import Session
from pathlib import Path
import json

# Prepare test file
test_file = Path("tests/system_linting_example.py")
test_file.parent.mkdir(parents=True, exist_ok=True)
test_file.write_text("def greet( name ):\n return f\"Hello, {name}!\"\n", encoding="utf-8")

# Define system input
system_input = {
    "file_path": str(test_file),
    "session_id": "system-test-session",
    "system": "linting"
}

# 🔁 Run SystemProvider (ID = 1)
system = SystemProviderFactory.create(id=1)
final_state = system.run(system_input)

print("🧪 Final System State:")
print(json.dumps(final_state.model_dump(), indent=2))

# 📚 Show recent transitions
print("\n📚 System Transitions:")
with Session(bind=engine) as session:
    logs = session.query(StateTransitionLog).filter_by(session_id="system-test-session").order_by(StateTransitionLog.timestamp).all()
    for log in logs:
        print(f"[{log.timestamp}] {log.from_state} → {log.to_state} | Reason: {log.reason}")


🧪 Final System State:
{
  "state": "end",
  "previous_state": "code_stability",
  "state_type": "end",
  "reason": "success",
  "decision": "final",
  "steps": 3,
  "max_steps": 20,
  "summary": "initial stability failed\u2014rejecting",
  "output": {
    "state": "end",
    "input_file": "tests\\system_linting_example.py",
    "working_file": "tests\\system_linting_example_working.py",
    "file_path": "tests\\system_linting_example.py",
    "session_id": "system-test-session",
    "system": "linting",
    "reason": "initial stability failed\u2014rejecting",
    "retry_count": 0,
    "_last_state": "code_stability",
    "steps": 1,
    "output": {
      "state": "end",
      "previous_state": "code_stability",
      "state_type": "end",
      "reason": "success",
      "decision": "final",
      "steps": 2,
      "max_steps": 20,
      "summary": "stability failed",
      "output": {
        "state": "end",
        "input_file": "tests\\system_linting_example.py",
        "working_fil

### 🔍 Step 11: Test Controller Provider

In [11]:
# 🧪 PreprocessingControllerProvider Test
from app.factories.controller_provider_factory import ControllerProviderFactory
from app.db.models import StateTransitionLog
from sqlalchemy.orm import Session
from pathlib import Path
import json

# 1) Prepare a small file
test_file = Path("tests/preprocess_only.py")
test_file.parent.mkdir(exist_ok=True, parents=True)
test_file.write_text("def foo(): return 42\n", encoding="utf-8")

# 2) Controller input
controller_input = {
    "file_name":   str(test_file),
    "session_id":  "ctrl-preproc-test",
    "system":      "preprocessing_controller"
}

# 3) Run the controller
ctrl = ControllerProviderFactory.create(id=1)
final = ctrl.run(controller_input)

print("🧪 Final Controller State:")
print(json.dumps(final.model_dump(), indent=2))

# 4) Inspect transitions
print("\n📚 Controller Transitions:")
with Session(bind=engine) as sess:
    logs = (
        sess.query(StateTransitionLog)
            .filter_by(session_id="ctrl-preproc-test")
            .order_by(StateTransitionLog.timestamp)
            .all()
    )
    for log in logs:
        print(f"{log.from_state} → {log.to_state}  ({log.reason})")


🧪 Final Controller State:
{
  "state": "end",
  "previous_state": "preprocess",
  "state_type": "end",
  "reason": "success",
  "decision": "final",
  "steps": 2,
  "max_steps": 20,
  "summary": "after preprocessing",
  "output": {
    "state": "end",
    "file_path": "tests\\preprocess_only.py",
    "working_file": "tests\\preprocess_only_working.py",
    "file_name": "tests\\preprocess_only.py",
    "session_id": "ctrl-preproc-test",
    "system": "preprocessing_controller",
    "steps": 2,
    "reason": "after preprocessing",
    "_last_state": "preprocess",
    "output": {
      "state": "end",
      "previous_state": "code_stability",
      "state_type": "end",
      "reason": "success",
      "decision": "final",
      "steps": 3,
      "max_steps": 20,
      "summary": "initial stability failed\u2014rejecting",
      "provider_name": "linting_system_provider"
    }
  },
  "provider_name": "preprocessing_controller"
}

📚 Controller Transitions:
start → preprocess  (kickoff)
start

### 🔍 Step 12: Test Program Provider

In [12]:
# 🧪 CodeCritic ProgramProvider Test

from app.factories.program_provider_factory import ProgramProviderFactory
from app.db.models import StateTransitionLog
from sqlalchemy.orm import Session
from pathlib import Path
import json

# 1. Prepare test file
test_file = Path("tests/codecritic_program_example.py")
test_file.parent.mkdir(parents=True, exist_ok=True)
test_file.write_text("def add( x, y ):\n return x+y\n", encoding="utf-8")

# 2. Prepare input
program_input = {
    "file_name":  str(test_file),
    "session_id": "program-test-session",
    "system":     "codecritic_program"
}

# 3. Run the program provider (ID = 1)
program = ProgramProviderFactory.create(id=1)
result = program.run(program_input)

# 4. Print final output
print("🧪 Final Program State:")
print(json.dumps(result.model_dump(), indent=2))

# 5. View transition logs
print("\n📚 Program Transitions:")
with Session(bind=engine) as sess:
    logs = (
        sess.query(StateTransitionLog)
        .filter_by(session_id="program-test-session")
        .order_by(StateTransitionLog.timestamp)
        .all()
    )
    for log in logs:
        print(f"{log.from_state} → {log.to_state}  ({log.reason})")


🧪 Final Program State:
{
  "state": "end",
  "previous_state": "preprocessing",
  "state_type": "end",
  "reason": "success",
  "decision": "final",
  "steps": 2,
  "max_steps": 20,
  "summary": "preprocessing complete",
  "output": {
    "state": "end",
    "file_name": "tests\\codecritic_program_example.py",
    "working_file": "tests\\codecritic_program_example_working.py",
    "session_id": "program-test-session",
    "system": "codecritic_program",
    "reason": "preprocessing complete",
    "_last_state": "preprocessing",
    "steps": 1,
    "output": {
      "state": "end",
      "previous_state": "preprocess",
      "state_type": "end",
      "reason": "success",
      "decision": "final",
      "steps": 2,
      "max_steps": 20,
      "summary": "after preprocessing",
      "provider_name": "preprocessing_controller"
    }
  },
  "provider_name": "codecritic_program"
}

📚 Program Transitions:
start → preprocessing  (start of program)
start → preprocess  (kickoff)
start → code_

### 🔍 Step 13: Test Session Execution

In [13]:
# 🧪 Run Program Using SessionConfig

from app.db.models import SessionConfig
from app.factories.program_provider_factory import ProgramProviderFactory
from sqlalchemy.orm import Session
from pathlib import Path
import json

# Fetch session config
with Session(bind=engine) as db:
    session_row = db.query(SessionConfig).filter_by(id=1).first()
    assert session_row, "❌ No session config found"

# Prepare test file
file = Path("tests/session_linked_execution.py")
file.write_text("def ping( user ):\n return f\"pong {user}\"")

# Construct input using session metadata
input_data = {
    "file_name":  str(file),
    "session_id": session_row.id,
    "system":     session_row.name
}

# Run associated program
program = ProgramProviderFactory.create(id=session_row.program_provider_id)
result = program.run(input_data)

print("✅ Final Program Result (via SessionConfig):")
print(json.dumps(result.model_dump(), indent=2))


✅ Final Program Result (via SessionConfig):
{
  "state": "end",
  "previous_state": "preprocessing",
  "state_type": "end",
  "reason": "success",
  "decision": "final",
  "steps": 2,
  "max_steps": 20,
  "summary": "preprocessing complete",
  "output": {
    "state": "end",
    "file_name": "tests\\session_linked_execution.py",
    "working_file": "tests\\session_linked_execution_working.py",
    "session_id": 1,
    "system": "codecritic_test_session",
    "reason": "preprocessing complete",
    "_last_state": "preprocessing",
    "steps": 1,
    "output": {
      "state": "end",
      "previous_state": "preprocess",
      "state_type": "end",
      "reason": "success",
      "decision": "final",
      "steps": 2,
      "max_steps": 20,
      "summary": "after preprocessing",
      "provider_name": "preprocessing_controller"
    }
  },
  "provider_name": "codecritic_program"
}
